# ADA 3 hybrid causal decoder

This notebook packages the reference PyTorch definitions from
`Ada_3/ada_v3.py` (SHA-256 `e93ffc1c3cc72b43b4951623441f5f831a0c2c57f5a064133a1e9c1f1c42d04b`) at tiny CPU scale. It covers
the design discussed in *ADA A First Experimental Hybrid Architecture
Combining Selective State-Space Models, Grouped-Query Attention, and
Mixture-of-Experts*. The paper describes a preliminary architecture
experiment. The executed fixture below is separate evidence.

The configurable stack alternates three state-space blocks with a
grouped-query attention and routed MoE block. A tied vocabulary projection
predicts the next token. Two auxiliary heads predict two and three tokens
ahead. The source's top-k MoE evaluates every expert, then combines selected
outputs. Count that separately from actual sparse execution.

## Run this notebook

Use Python 3.11 or later, JupyterLab and a CPU PyTorch build. In an isolated virtual
environment, install `jupyterlab`, `ipykernel` and `torch>=2.6`, then open this
notebook and run all cells. The reference run is deterministic on CPU and neither
reads repository data nor downloads models. Training starts from random weights.

In [1]:
import json
import math
from dataclasses import asdict, dataclass, field, replace
from typing import Any, Callable, Dict, List, Optional, Tuple, Union

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

SEED = 2031
torch.manual_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
DEVICE = torch.device("cpu")
print(f"PyTorch {torch.__version__}, device={DEVICE}, seed={SEED}")

PyTorch 2.11.0+cpu, device=cpu, seed=2031


## Source definitions

The cells preserve the v3 CPU arithmetic. Only the isolated fixture and evaluation are new.

In [2]:
@dataclass
class ADAConfig:
    # ── vocabulary ────────────────────────────────────────────────────────────
    vocab_size: int       = 50_272        # GPT-NeoX / cl100k tokenizer

    # ── core dimensions ───────────────────────────────────────────────────────
    d_model: int          = 2_048
    n_heads: int          = 16
    n_kv_heads: int       = 4             # GQA: 4 query heads share 1 KV head
    n_layers: int         = 32

    # ── Mamba-2 SSM ───────────────────────────────────────────────────────────
    mamba_d_state: int    = 128           # N: state dimension per SSM head
    mamba_d_conv: int     = 4             # causal depthwise conv kernel width
    mamba_expand: int     = 2             # d_inner = expand × d_model
    mamba_n_heads: int    = 64            # H: SSM parallelism heads (d_inner / H = hd)
    mamba_dt_min: float   = 0.001
    mamba_dt_max: float   = 0.1

    # ── dense SwiGLU FFN (used inside GQA blocks) ─────────────────────────────
    d_ff: int             = 5_504         # ⌊(8/3) × d_model⌋  same param budget as 4×FFN

    # ── Fine-Grained MoE ──────────────────────────────────────────────────────
    n_experts: int        = 64
    n_active: int         = 6             # top-k experts routed per token
    moe_d_ff: int         = 1_408         # per-expert SwiGLU intermediate dim
    bias_lr: float        = 1e-3          # load-balance bias update rate

    # ── Mixture of Depths ─────────────────────────────────────────────────────
    mod_capacity: float   = 0.5           # fraction of tokens through each Mamba layer

    # ── Multi-Token Prediction ────────────────────────────────────────────────
    n_mtp: int            = 4             # predict tokens at t+2, t+3, t+4, t+5

    # ── misc ──────────────────────────────────────────────────────────────────
    max_seq_len: int      = 4_096
    dropout: float        = 0.0
    rope_theta: float     = 500_000.0    # LLaMA-3 long-context θ
    norm_eps: float       = 1e-6

    # ── derived (read-only) ───────────────────────────────────────────────────
    @property
    def head_dim(self) -> int:
        assert self.d_model % self.n_heads == 0, "d_model must be divisible by n_heads"
        return self.d_model // self.n_heads

    @property
    def mamba_inner(self) -> int:
        return self.mamba_expand * self.d_model

    @property
    def mamba_head_dim(self) -> int:
        di = self.mamba_inner
        assert di % self.mamba_n_heads == 0, (
            f"mamba_n_heads={self.mamba_n_heads} must divide mamba_inner={di}"
        )
        return di // self.mamba_n_heads

    @property
    def block_pattern(self) -> List[str]:
        """3× Mamba(MoD) + 1× MoE-Attn, cycled over n_layers."""
        base = ["mamba", "mamba", "mamba", "moe_attn"]
        return (base * (self.n_layers // 4 + 1))[: self.n_layers]


class RMSNorm(nn.Module):
    """Root-mean-square layer normalisation. No additive bias."""

    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps    = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        fp32 = x.float()
        norm = fp32.pow(2).mean(dim=-1, keepdim=True).add(self.eps).rsqrt()
        return (fp32 * norm).to(x.dtype) * self.weight


def build_rope_cache(
    seq_len: int,
    head_dim: int,
    theta: float = 10_000.0,
    device: Optional[torch.device] = None,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Precompute RoPE cos/sin tables.
    Returns (cos, sin) each of shape (seq_len, head_dim // 2), float32.
    """
    half     = head_dim // 2
    inv_freq = 1.0 / (theta ** (torch.arange(0, half, device=device).float() / half))
    t        = torch.arange(seq_len, device=device, dtype=torch.float32)
    freqs    = torch.outer(t, inv_freq)           # (seq_len, half)
    return freqs.cos(), freqs.sin()


def apply_rope(
    x: torch.Tensor,
    cos: torch.Tensor,
    sin: torch.Tensor,
    position_ids: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """
    Apply rotary position embeddings.

    x            : (B, n_heads, T, head_dim)
    cos, sin     : (max_seq_len, head_dim // 2) — precomputed cache
    position_ids : (B, T) or None — enables non-contiguous positions (MoD, KV-cache)
    """
    T    = x.shape[2]
    half = x.shape[-1] // 2

    if position_ids is not None:
        c = cos[position_ids].unsqueeze(1)     # (B, 1, T, half)
        s = sin[position_ids].unsqueeze(1)
    else:
        c = cos[:T].unsqueeze(0).unsqueeze(0)  # (1, 1, T, half)
        s = sin[:T].unsqueeze(0).unsqueeze(0)

    x1, x2 = x[..., :half], x[..., half:]
    return torch.cat([x1 * c - x2 * s, x2 * c + x1 * s], dim=-1)


class SwiGLU(nn.Module):
    """
    Gated MLP: out = W_down( SiLU(W_gate(x)) ⊙ W_up(x) ).
    Three weight matrices; set d_ff = (2/3) × 4 × d_model to keep the same
    parameter count as a standard 2-matrix FFN with d_ff = 4 × d_model.
    """

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.0):
        super().__init__()
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w_up   = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff,    d_model, bias=False)
        self.drop   = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w_down(self.drop(F.silu(self.w_gate(x)) * self.w_up(x)))

In [3]:
class GQAAttention(nn.Module):
    """
    Grouped Query Attention with pre-cached RoPE and causal mask.

    n_kv_heads < n_heads: each KV head is shared by (n_heads // n_kv_heads) Q heads.
    RoPE cache is a persistent non-parameter buffer — correct device placement
    via .to() and torch.compile-safe (no dynamic recompute in forward).
    """

    def __init__(self, cfg: ADAConfig):
        super().__init__()
        self.n_heads  = cfg.n_heads
        self.n_kv     = cfg.n_kv_heads
        self.head_dim = cfg.head_dim
        self.n_rep    = cfg.n_heads // cfg.n_kv_heads   # broadcast factor
        self.scale    = cfg.head_dim ** -0.5

        D, H, Hkv, hd = cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.head_dim
        self.wq  = nn.Linear(D, H   * hd, bias=False)
        self.wk  = nn.Linear(D, Hkv * hd, bias=False)
        self.wv  = nn.Linear(D, Hkv * hd, bias=False)
        self.wo  = nn.Linear(H * hd, D,   bias=False)
        self.drop = nn.Dropout(cfg.dropout)

        cos, sin = build_rope_cache(cfg.max_seq_len, cfg.head_dim, cfg.rope_theta)
        # persistent=False: not saved in state_dict; recomputed from config on load
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def forward(
        self,
        x: torch.Tensor,
        position_ids: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        B, T, _ = x.shape
        hd = self.head_dim

        q = self.wq(x).reshape(B, T, self.n_heads, hd).transpose(1, 2)  # (B,H,T,hd)
        k = self.wk(x).reshape(B, T, self.n_kv,   hd).transpose(1, 2)  # (B,Hkv,T,hd)
        v = self.wv(x).reshape(B, T, self.n_kv,   hd).transpose(1, 2)

        q = apply_rope(q, self.rope_cos, self.rope_sin, position_ids)
        k = apply_rope(k, self.rope_cos, self.rope_sin, position_ids)

        if self.n_rep > 1:                                               # GQA broadcast
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale      # (B,H,T,T)
        causal = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
        scores = scores.masked_fill(~causal[None, None], float("-inf"))

        # Softmax in float32 for numerical stability; cast back to x.dtype
        attn = F.softmax(scores.float(), dim=-1).to(x.dtype)
        attn = self.drop(attn)

        out = torch.matmul(attn, v).transpose(1, 2).reshape(B, T, -1)  # (B,T,D)
        return self.wo(out)


class Mamba2SSM(nn.Module):
    def __init__(self, cfg: ADAConfig):
        super().__init__()
        di = cfg.mamba_inner        # expand × d_model
        N  = cfg.mamba_d_state
        H  = cfg.mamba_n_heads
        hd = cfg.mamba_head_dim     # di // H

        self.n_heads  = H
        self.head_dim = hd
        self.d_state  = N
        self.d_inner  = di

        # Single fused projection: z(di) + x(di) + B(N) + C(N) + dt(H)
        self.in_proj = nn.Linear(cfg.d_model, 2 * di + 2 * N + H, bias=False)

        # Causal depthwise conv on the x branch.
        # padding = d_conv - 1 on both sides; we trim the right side to length L.
        self.conv1d = nn.Conv1d(
            di, di,
            kernel_size=cfg.mamba_d_conv,
            groups=di,
            padding=cfg.mamba_d_conv - 1,
            bias=True,
        )

        # dt bias: initialised from log-uniform(dt_min, dt_max).
        # Stored as log(dt_init); softplus in forward ≈ dt_init for small values.
        log_dt = (
            torch.rand(H) * (math.log(cfg.mamba_dt_max) - math.log(cfg.mamba_dt_min))
            + math.log(cfg.mamba_dt_min)
        )
        self.dt_bias = nn.Parameter(log_dt)

        # A_log: A[h] = -exp(A_log[h]) < 0. Init: log(1 … H) → A ≈ -1 … -H.
        self.A_log = nn.Parameter(
            torch.log(torch.arange(1, H + 1, dtype=torch.float32))
        )

        # D: per-head skip-connection scalar.
        self.D = nn.Parameter(torch.ones(H))

        # Post-scan normalisation + gating (Mamba-2 training stability).
        self.out_norm = RMSNorm(di, cfg.norm_eps)
        self.out_proj = nn.Linear(di, cfg.d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, _ = x.shape
        H, hd, N, di = self.n_heads, self.head_dim, self.d_state, self.d_inner

        # ── fused input projection ───────────────────────────────────────────
        proj = self.in_proj(x)                                   # (B, L, 2di+2N+H)
        z, x_in, B_in, C_in, dt = proj.split([di, di, N, N, H], dim=-1)
        # z    : (B, L, di)  — output gate
        # x_in : (B, L, di)  — main SSM input
        # B_in : (B, L, N)   — input projection, shared across heads
        # C_in : (B, L, N)   — output projection, shared across heads
        # dt   : (B, L, H)   — per-head time-step logits

        # ── causal depthwise conv ────────────────────────────────────────────
        # Input to conv: (B, di, L). Padding adds d_conv-1 on each side.
        # Trimming to [:, :, :L] keeps only the causal (left-padded) outputs.
        x_in = F.silu(
            self.conv1d(x_in.transpose(1, 2))[..., :L]          # (B, di, L)
        ).transpose(1, 2)                                         # (B, L, di)

        # ── reshape to per-head tensors ──────────────────────────────────────
        x_h = x_in.reshape(B, L, H, hd)                         # (B, L, H, hd)

        # ── discretize A and B ───────────────────────────────────────────────
        A    = -torch.exp(self.A_log.float())                    # (H,) negative
        dt   = F.softplus(dt.float() + self.dt_bias)            # (B, L, H) positive
        Abar = torch.exp(dt * A)                                 # (B, L, H) ∈ (0,1)
        # B̄ₜ[h] = dtₜ[h] · Bₜ  →  broadcast to (B, L, H, N)
        Bbar = dt.unsqueeze(-1) * B_in.float().unsqueeze(2)     # (B, L, H, N)

        # ── selective scan (sequential; see module docstring for production) ──
        h   = x.new_zeros(B, H, hd, N, dtype=torch.float32)
        ys  = x.new_empty(B, L, H, hd)

        for t in range(L):
            # a_t: (B, H, 1, 1) — scalar decay per head, broadcast to state
            a_t = Abar[:, t, :, None, None]
            # b_t: (B, H, 1, N) — input projection, broadcast over hd
            b_t = Bbar[:, t, :, None, :]
            # u_t: (B, H, hd, 1) — per-head input, broadcast over N
            u_t = x_h[:, t, :, :, None].float()

            h = a_t * h + b_t * u_t                              # (B, H, hd, N)

            # y = (C · h).sum(N) + D · x_skip
            c_t    = C_in[:, t, None, None, :].float()           # (B, 1, 1, N)
            y_skip = self.D[None, :, None] * x_h[:, t].float()  # (B, H, hd)
            ys[:, t] = ((h * c_t).sum(-1) + y_skip).to(x.dtype)

        # ── gate + project ───────────────────────────────────────────────────
        y = self.out_norm(ys.reshape(B, L, di))
        y = y * F.silu(z)
        return self.out_proj(y)


class Mamba2Block(nn.Module):
    """
    Pre-norm Mamba-2 block. Does NOT add its own residual connection.
    The residual is applied by the MoD wrapper as:
        x_out[sel] = x[sel] + router_weight[sel] · Mamba2Block(x[sel])
    This is the correct MoD formulation (Raposo et al. §3).
    """

    def __init__(self, cfg: ADAConfig):
        super().__init__()
        self.norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ssm  = Mamba2SSM(cfg)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.ssm(self.norm(x))

In [4]:
class Expert(nn.Module):
    """Single SwiGLU expert. No dropout — experts see sparse, token-subset inputs."""

    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w_up   = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff,    d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


class FineGrainedMoE(nn.Module):
    def __init__(self, cfg: ADAConfig):
        super().__init__()
        self.n_experts = cfg.n_experts
        self.n_active  = cfg.n_active
        self.bias_lr   = cfg.bias_lr

        self.router  = nn.Linear(cfg.d_model, cfg.n_experts, bias=False)
        self.experts = nn.ModuleList([
            Expert(cfg.d_model, cfg.moe_d_ff) for _ in range(cfg.n_experts)
        ])
        # Not a Parameter → excluded from optimizer; updated manually below.
        self.register_buffer("expert_bias", torch.zeros(cfg.n_experts))

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Returns
        -------
        out  : (B, T, d_model)
        load : (n_experts,) fraction of tokens routed to each expert
        """
        B, T, D = x.shape
        n_tok   = B * T
        x_flat  = x.reshape(n_tok, D)

        logits = self.router(x_flat)                             # (n_tok, E)
        biased = logits + self.expert_bias                       # routing signal only

        _, topk_idx = torch.topk(biased, self.n_active, dim=-1) # (n_tok, k)

        # Weights from unbiased logits — gradient flows here, not through bias
        weights = F.softmax(
            logits.gather(1, topk_idx).float(), dim=-1
        ).to(x.dtype)                                            # (n_tok, k)

        # Dense dispatch: (n_tok, E, D) — all experts run
        expert_outs = torch.stack(
            [exp(x_flat) for exp in self.experts], dim=1
        )                                                         # (n_tok, E, D)

        # Build combine weight matrix: (n_tok, E), sparse over k active experts
        combine = torch.zeros(n_tok, self.n_experts, device=x.device, dtype=x.dtype)
        combine.scatter_(1, topk_idx, weights)

        out = torch.einsum("te,ted->td", combine, expert_outs)   # (n_tok, D)

        # Online bias update — no gradients, no loss term, training only
        load = torch.zeros(self.n_experts, device=x.device, dtype=torch.float32)
        if self.training:
            with torch.no_grad():
                load.scatter_add_(
                    0,
                    topk_idx.flatten(),
                    torch.ones(n_tok * self.n_active, device=x.device),
                )
                load_frac = load / max(1, n_tok * self.n_active)
                self.expert_bias.add_(
                    self.bias_lr * (1.0 / self.n_experts - load_frac)
                )
                load = load_frac
        else:
            # Eval: use combine weights as a proxy for load
            load = combine.detach().sum(0)

        return out.reshape(B, T, D), load


class MoEAttnBlock(nn.Module):
    """
    Full transformer block: pre-norm GQA + residual, then pre-norm MoE FFN + residual.
    Residual connections live inside this block (unlike Mamba2Block, which delegates to MoD).
    """

    def __init__(self, cfg: ADAConfig):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn      = GQAAttention(cfg)
        self.ffn_norm  = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.moe       = FineGrainedMoE(cfg)

    def forward(
        self,
        x: torch.Tensor,
        position_ids: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        x = x + self.attn(self.attn_norm(x), position_ids)
        moe_out, load = self.moe(self.ffn_norm(x))
        x = x + moe_out
        return x, load


class MoD(nn.Module):
    def __init__(self, block: nn.Module, d_model: int, capacity: float):
        super().__init__()
        self.block    = block
        self.capacity = capacity
        self.router   = nn.Linear(d_model, 1, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, D = x.shape
        k = max(1, int(T * self.capacity))

        scores = self.router(x).squeeze(-1)                      # (B, T)

        # Top-k by router score (unsorted — we sort by position next)
        _, top_idx = torch.topk(scores, k, dim=-1, sorted=False) # (B, k)

        # Router weight: sigmoid of the score at each selected position
        weights = torch.sigmoid(scores.gather(1, top_idx))       # (B, k)

        # Sort by position index — mandatory for causal SSM correctness
        top_idx_sorted, perm = top_idx.sort(dim=-1)              # (B, k)
        weights_sorted       = weights.gather(1, perm)            # (B, k)

        # Gather the selected token representations
        idx_exp = top_idx_sorted.unsqueeze(-1).expand(B, k, D)
        x_sel   = x.gather(1, idx_exp)                           # (B, k, D)

        # Block processes selected tokens (returns contribution, not full residual)
        block_out = self.block(x_sel)                             # (B, k, D)

        # Weighted residual: selected positions get x + weight · block(x)
        x_proc = x_sel + weights_sorted.unsqueeze(-1) * block_out

        # Scatter back; unselected positions retain their original x values
        out = x.clone()
        out.scatter_(1, idx_exp, x_proc)
        return out


class ADA(nn.Module):
    """
    Adaptive Depth Architecture v3.0

    Layer topology (original 32-layer preset, block_pattern property):
      0, 1, 2  →  MoD( Mamba2Block )   token-selective linear SSM
      3        →  MoEAttnBlock          global GQA + fine-grained MoE FFN
      4, 5, 6  →  MoD( Mamba2Block )
      7        →  MoEAttnBlock
      … × 8 cycles (layers 0–31)

    Loss function:
      L_total = L_CE(t+1) + 0.3 × mean_i( L_CE(t+i+2) )
      No auxiliary MoE loss — bias update handles load balancing.
    """

    def __init__(self, cfg: ADAConfig):
        super().__init__()
        self.cfg = cfg

        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop  = nn.Dropout(cfg.dropout)

        self.layers: nn.ModuleList = nn.ModuleList()
        for bt in cfg.block_pattern:
            if bt == "mamba":
                self.layers.append(
                    MoD(Mamba2Block(cfg), cfg.d_model, cfg.mod_capacity)
                )
            elif bt == "moe_attn":
                self.layers.append(MoEAttnBlock(cfg))
            else:
                raise ValueError(f"Unknown block type: {bt!r}")

        self.norm    = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        # Weight tying: share embedding and output projection weights.
        # Saves vocab_size × d_model parameters; standard for LMs.
        self.lm_head.weight = self.embed.weight

        # Multi-Token Prediction heads: head i predicts token at t + i + 2.
        # Head 0 → t+2, head 1 → t+3, …, head n_mtp-1 → t+n_mtp+1.
        self.mtp_heads = nn.ModuleList([
            nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
            for _ in range(cfg.n_mtp)
        ])

        self._init_weights()

    def _init_weights(self):
        std = 0.02
        nn.init.trunc_normal_(self.embed.weight, std=std)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.trunc_normal_(module.weight, std=std)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Conv1d):
                nn.init.trunc_normal_(module.weight, std=std)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, RMSNorm):
                nn.init.ones_(module.weight)

    def forward(
        self,
        input_ids: torch.Tensor,
        position_ids: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, List[torch.Tensor], List[torch.Tensor]]:
        """
        Parameters
        ----------
        input_ids   : (B, T)  token IDs
        position_ids: (B, T)  or None — defaults to 0…T-1

        Returns
        -------
        logits       : (B, T, vocab_size)           next-token prediction
        mtp_logits   : list of (B, T, vocab_size)   one per MTP head
        expert_loads : list of (n_experts,)          one per MoE layer
        """
        B, T = input_ids.shape
        if position_ids is None:
            position_ids = (
                torch.arange(T, device=input_ids.device)
                .unsqueeze(0)
                .expand(B, -1)
            )

        x = self.drop(self.embed(input_ids))              # (B, T, d_model)

        expert_loads: List[torch.Tensor] = []
        for layer in self.layers:
            if isinstance(layer, MoEAttnBlock):
                x, load = layer(x, position_ids)
                expert_loads.append(load)
            else:                                          # MoD(Mamba2Block)
                x = layer(x)

        x = self.norm(x)

        logits     = self.lm_head(x)                      # (B, T, V)
        mtp_logits = [head(x) for head in self.mtp_heads]

        return logits, mtp_logits, expert_loads

    def compute_loss(
        self,
        input_ids: torch.Tensor,
        position_ids: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        """
        Unified training loss.

        L = L_main + 0.3 × mean( L_mtp )

        L_main: cross-entropy predicting t+1 from t  (standard LM loss)
        L_mtp:  mean cross-entropy over MTP heads
                head i predicts token at t+i+2 from hidden at t
        MoE:    no auxiliary loss — bias update handles load balancing
        """
        B, T = input_ids.shape
        V    = self.cfg.vocab_size

        logits, mtp_logits, expert_loads = self.forward(input_ids, position_ids)

        # Main: logits[:, :-1] → predict input_ids[:, 1:]
        main_loss = F.cross_entropy(
            logits[:, :-1].reshape(-1, V),
            input_ids[:, 1:].reshape(-1),
        )

        # MTP: head i uses logits[:, :T-offset] to predict input_ids[:, offset:]
        mtp_loss = torch.tensor(0.0, device=input_ids.device)
        n_valid  = 0
        for i, head_logits in enumerate(mtp_logits):
            offset = i + 2
            if T - offset <= 0:
                continue
            mtp_loss = mtp_loss + F.cross_entropy(
                head_logits[:, : T - offset].reshape(-1, V),
                input_ids[:, offset:].reshape(-1),
            )
            n_valid += 1
        if n_valid > 0:
            mtp_loss = mtp_loss / n_valid

        total = main_loss + 0.3 * mtp_loss

        # Expert balance entropy (monitoring metric, not optimised)
        load_ent = 0.0
        if expert_loads:
            for ld in expert_loads:
                p = ld.float() / (ld.sum() + 1e-9)
                load_ent += -(p * (p + 1e-9).log()).sum().item()
            load_ent /= len(expert_loads)

        return total, {
            "loss":         total.item(),
            "loss_main":    main_loss.item(),
            "loss_mtp":     mtp_loss.item(),
            "load_entropy": load_ent,
        }

## Tiny training fixture

Each sequence uses a fixed step modulo 23. A separate RNG stream generates
evaluation rows. Setting the depth capacity to one keeps all positions in
sequence order while preserving the source's sigmoid residual weighting.
The reduced capacity is tested separately below.

In [5]:
cfg = ADAConfig(vocab_size=23, d_model=32, n_layers=4, n_heads=4,
                n_kv_heads=2, mamba_d_state=4, mamba_n_heads=4,
                d_ff=64, n_experts=4, n_active=2, moe_d_ff=24,
                n_mtp=2, max_seq_len=32, dropout=0.0, mod_capacity=1.0)
build_model = ADA

In [6]:
def arithmetic_sequences(seed, count, length=12, vocab_size=23):
    rng = torch.Generator().manual_seed(seed)
    start = torch.randint(0, vocab_size, (count, 1), generator=rng)
    step = torch.randint(1, 5, (count, 1), generator=rng)
    return (start + step * torch.arange(length)) % vocab_size


train_rows = arithmetic_sequences(SEED + 1, 16)
eval_rows = arithmetic_sequences(SEED + 2, 4)
assert train_rows.dtype == torch.long and eval_rows.shape == (4, 12)
print("Synthetic train/eval shapes:", tuple(train_rows.shape), tuple(eval_rows.shape))

Synthetic train/eval shapes: (16, 12) (4, 12)


In [7]:
assert cfg.block_pattern == ["mamba", "mamba", "mamba", "moe_attn"]
model = build_model(cfg).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=0.002)


@torch.inference_mode()
def evaluate(ids):
    model.eval()
    logits = model(ids)[0][:, :-1]
    next_ids = ids[:, 1:]
    ce = F.cross_entropy(logits.flatten(0, 1), next_ids.flatten())
    accuracy = (logits.argmax(-1) == next_ids).float().mean()
    return float(ce), float(accuracy)


before, _ = evaluate(eval_rows)
model.train()
metrics = None
for step in range(8):
    batch = train_rows[(step % 4) * 4 : (step % 4 + 1) * 4]
    opt.zero_grad(set_to_none=True)
    loss, metrics = model.compute_loss(batch)
    loss.backward()
    norm = nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    assert torch.isfinite(loss) and torch.isfinite(norm) and norm > 0
    opt.step()
after, accuracy = evaluate(eval_rows)
assert math.isfinite(after) and 0.0 <= accuracy <= 1.0

with torch.inference_mode():
    ids = eval_rows[:1, :4].clone()
    for _ in range(3):
        logits, auxiliary, loads = model(ids)
        assert logits.shape == (*ids.shape, cfg.vocab_size)
        assert len(auxiliary) == cfg.n_mtp
        assert len(loads) == 1 and torch.isfinite(loads[0]).all()
        ids = torch.cat((ids, logits[:, -1].argmax(dim=-1, keepdim=True)), dim=1)
    assert ids.shape == (1, 7)

print(json.dumps({
    "fixture": "fixed-step modulo arithmetic",
    "parameter_count": sum(parameter.numel() for parameter in model.parameters()),
    "validation_targets": eval_rows[:, 1:].numel(),
    "before_ce": round(before, 4),
    "after_ce": round(after, 4),
    "accuracy": round(accuracy, 4),
    "train_last": {name: round(value, 4) for name, value in metrics.items()},
    "sample_ids": ids[0].tolist(),
}, indent=2))

{
  "fixture": "fixed-step modulo arithmetic",
  "parameter_count": 35684,
  "validation_targets": 44,
  "before_ce": 3.1711,
  "after_ce": 2.7781,
  "accuracy": 0.4091,
  "train_last": {
    "loss": 3.5667,
    "loss_main": 2.6845,
    "loss_mtp": 2.9406,
    "load_entropy": 1.3748
  },
  "sample_ids": [
    19,
    22,
    2,
    5,
    8,
    8,
    12
  ]
}


## Prefix parity and routing failure

Compare both the SSM block and entire model with a mutated suffix and
a truncated sequence. Then test the original MoD rule at capacity 0.5
on a controlled score example. Prefix changes under that rule block
claims that the historical route is a causal decoder.

In [8]:
original = eval_rows[:2].clone()
altered_suffix = original.clone()
cut = 6
altered_suffix[:, cut:] = (altered_suffix[:, cut:] + 10) % cfg.vocab_size
with torch.inference_mode():
    full, aux, _ = model(original)
    altered = model(altered_suffix)[0]
    truncated = model(original[:, :cut])[0]
    input_features = model.embed(original)
    ssm = model.layers[0].block
    ssm_full = ssm(input_features)
    ssm_prefix = ssm(input_features[:, :cut])
    torch.testing.assert_close(full[:, :cut], altered[:, :cut], rtol=1e-5, atol=1e-5)
    torch.testing.assert_close(full[:, :cut], truncated, rtol=1e-5, atol=1e-5)
    torch.testing.assert_close(ssm_full[:, :cut], ssm_prefix, rtol=1e-5, atol=1e-5)
    assert all(logits.shape == full.shape for logits in aux)
print("PASS: SSM and full decoder prefix consistency in the all-token routing control.")


def mod_ranking_counterexample():
    # Weight 1 on feature 0: token 1 wins the 50% budget at length 2,
    # but appending a higher score at length 3 displaces it.
    class FixedContribution(nn.Module):
        def forward(self, inputs):
            return torch.ones_like(inputs)

    test_router = MoD(FixedContribution(), 1, 0.5).eval()
    with torch.inference_mode():
        test_router.router.weight.fill_(1)
        if test_router.router.bias is not None:
            test_router.router.bias.zero_()
        inputs = torch.tensor([[[0.0], [1.0], [10.0]]])
        prefix = test_router(inputs[:, :2])
        full = test_router(inputs)
        return float((prefix - full[:, :2]).abs().max())


delta = mod_ranking_counterexample()
assert delta > 1e-3
print(f"EXPECTED FAILURE: historical 50% global top-k MoD is not prefix invariant (delta={delta:.6f}).")

PASS: SSM and full decoder prefix consistency in the all-token routing control.
EXPECTED FAILURE: historical 50% global top-k MoD is not prefix invariant (delta=0.731059).


## Verify the dense dispatch behavior

The v3 combine mask is sparse. The expert input tensors are deliberately
inspected to distinguish masking from skipped computation.

In [9]:
moe = model.layers[-1].moe
observed_counts = []
hooks = [expert.register_forward_hook(
    lambda _expert, args, _out: observed_counts.append(args[0].shape[0])
) for expert in moe.experts]
try:
    with torch.inference_mode():
        model(original)
finally:
    for hook in hooks:
        hook.remove()
assert observed_counts == [original.numel()] * cfg.n_experts
print(f"Dense v3 dispatch: expert/token evaluations={sum(observed_counts)}, "
      f"routed assignments={original.numel() * cfg.n_active}")

Dense v3 dispatch: expert/token evaluations=96, routed assignments=48


## Interpretation

The tiny run checks the local full-forward mathematical program and
dense routing behavior. It supplies no GPU performance estimate, cached
decoding parity, speculative decoding, or real language quality result.
Keep the measured MoD causality failure visible when comparing variants.